In [1]:
from new_module.dev_utils.utils import *
import pandas as pd
import re

pd.set_option('display.max_colwidth', None)

## Task: toxicity

### gpt3.5

In [4]:
# 문제점: gpt3.5 generation을 읽어보니 딱히 toxic하지가 않아서 고치더라도 그것이 그렇게 impactful 하지가 않다. 처음에 데이터셋 만들고 toxicity metric 뽑았을 때부터, 그게 어느 정도인지 눈으로 봤어야 하는데..

# read data
orig = read_outputs('/data/hyeryung/mucoco/new_module/llm_experiments/generate_with_llm/baselm_gens/gpt-3.5-turbo-0125/nontoxic/gpt-3.5-turbo-0125_realtoxicityprompts_0shot_150_below_nontoxic_threshold_0_95_332.jsonl')
llmle = read_outputs('/data/hyeryung/mucoco/outputs/llmedit/results_saehee_2024/qwen/2_tox_edited_38592.jsonl_total_0')
llmle_wo_orig = read_outputs('/data/hyeryung/mucoco/outputs/llmedit/results_2025/tox_iter_1_prompt_ablation/wo_orig/final/2_tox_loc_edit_50269.jsonl')
mlmle = read_outputs('/data/hyeryung/mucoco/outputs/toxicity/llm/j18pi8ab/outputs_epsilon0.95.txt.0')
llmwo = read_outputs('/data/hyeryung/mucoco/outputs/llmedit/results_saehee_2024/qwen/5_tox_loc_edit_38576.jsonl')
located = read_outputs('/data/hyeryung/mucoco/outputs/llmedit/dump/iter_loc_edit_qwen/located/2_tox_located_38592.jsonl_filtered_0')

agg_data = pd.DataFrame({
    'prompt': orig['prompt'],
    'original': orig['text'],
    'located': located['text'],
    'le_llm_edit': llmle['text'],
    'le_llm_edit_wo_orig': llmle_wo_orig['text'],
    'le_mask_infill': mlmle['text'],
    'llm_edit_wo_locate': llmwo['text'],
})

# add toxicity score
orig_toxicity = read_metric_file('/data/hyeryung/mucoco/new_module/llm_experiments/generate_with_llm/baselm_gens/gpt-3.5-turbo-0125/nontoxic/gpt-3.5-turbo-0125_realtoxicityprompts_0shot_150.jsonl-results.txt.toxicity', 
                 'toxicity')
agg_data['original_toxicity'] = orig_toxicity['toxicity']


In [4]:
# select samples with highest toxicity score within a prompt and select prompt-generation pairs with the highest toxicity score
agg_data = agg_data.sort_values('original_toxicity', ascending=False)
sampled_data = agg_data.groupby('prompt').head(1)
sampled_data = sampled_data.sort_values('original_toxicity', ascending=False)
sampled_data = sampled_data.iloc[:10].copy()

In [6]:
sampled_data.to_excel('/data/hyeryung/mucoco/outputs/20250323_generation_examples/toxicity_generation_examples.xlsx', index=False)

In [5]:
agg_data.to_excel('/data/hyeryung/mucoco/outputs/20250323_generation_examples/toxicity_generation_examples_all.xlsx', index=False)

In [ ]:

# # random sample 10 rows
# sampled_data = agg_data.groupby('prompt').sample(random_state=42, n=1)
# sampled_data = sampled_data.sample(random_state=42, n=10)

# check data


In [ ]:
# sampled_data.to_excel('/data/hyeryung/mucoco/outputs/20250323_generation_examples/toxicity_generation_examples.xlsx', index=False)

### gpt2

In [3]:
# read data
orig = read_outputs('/data/hyeryung/mucoco/new_module/data/toxicity-avoidance/testset_gpt2_2500.jsonl')
# llmle = read_outputs('')
mlmle = read_outputs('/data/hyeryung/mucoco/outputs/toxicity/llm/orywpep6/outputs_epsilon0.95.txt')
# llmwo = read_outputs('')
located = read_outputs('/data/hyeryung/mucoco/outputs/toxicity/llm/orywpep6/outputs_epsilon0.95.txt.intermediate')
mixmatch = read_outputs('/data/hyeryung/mixmatch/output_samples/detoxic_r/mask_disc_max_len_12_jigsaw_clsf_data_detoxic_em_max_iter_5_temp_1.0_shuffle_True_block_False_alpha_140.0_beta_1.0_delta_15.0_gamma_0.0_theta_100.0_date_08_06_2024_17_01_54/opt_samples.jsonl')
bolt = read_outputs('/data/hyeryung/BOLT/detoxic/detoxic/gen_len20.jsonl')
mucola = read_outputs('/data/hyeryung/mucoco/outputs/toxicity/mucola/gbi-word-netps-1-nls-1-os200-es40-allsat-toxic-to-nontoxic-attention-xo9cuxwt/outputs_epsilon-1.09861228867.txt')


In [ ]:

agg_data = pd.DataFrame({
    'prompt': orig['prompt'],
    'original': orig['text'],
    'located': located['iter0_masked_sentence'],
    # 'le_llm_edit': llmle['text'],
    'le_mask_infill': mlmle['text'],
    # 'llm_edit_wo_locate': llmwo['text'],
    'mixmatch': mixmatch['text'],
    'bolt': bolt['text'],
    'mucola': mucola['text'],
})

# add toxicity score
orig_toxicity = read_metric_file('/data/hyeryung/mucoco/new_module/data/toxicity-avoidance/results_gpt2.txt.toxicity', 
                 'toxicity')
agg_data['original_toxicity'] = orig_toxicity['toxicity']


In [6]:
# select samples with highest toxicity score within a prompt and select prompt-generation pairs with the highest toxicity score
agg_data = agg_data.sort_values('original_toxicity', ascending=False)
sampled_data = agg_data.groupby('prompt').head(1)
sampled_data = sampled_data.sort_values('original_toxicity', ascending=False)
sampled_data = sampled_data.iloc[:10].copy()

In [10]:
sampled_data['located'] = sampled_data['located'].apply(lambda x: re.sub(r"(<mask>)+", "<mask>", x))

In [12]:
sampled_data.to_excel('/data/hyeryung/mucoco/outputs/20250323_generation_examples/toxicity_generation_gpt2_examples.xlsx',index=False)

In [14]:
agg_data.to_excel('/data/hyeryung/mucoco/outputs/20250323_generation_examples/toxicity_generation_gpt2_examples_all.xlsx', index=False)

## Task: NLI

In [12]:
# read data
orig = read_outputs('/data/hyeryung/mucoco/new_module/data/logical-consistency/anli-r2-test_prompt_4_below_consistent_threshold_3105.jsonl')
llmle = read_outputs('/data/hyeryung/mucoco/outputs/llmedit/results_saehee_2024/qwen/6_nli_edited_38454.jsonl_total_0')
llmle_masked = read_outputs('/data/hyeryung/mucoco/outputs/llmedit/results_2025/nli_iter_1_prompt_ablation/wo_orig/edited/6_nli_edited_50967.jsonl_total_0')
mlmle = read_outputs('/data/hyeryung/mucoco/outputs/nli/ay8ohdbp/outputs_epsilon0.99.txt')
llmwo = read_outputs('/data/hyeryung/mucoco/outputs/llmedit/results_saehee_2024/qwen/9_nli_loc_edit_38546.jsonl')
located = read_outputs('/data/hyeryung/mucoco/outputs/llmedit/dump/iter_loc_edit_qwen/located/6_nli_located_38454.jsonl_filtered_0')

agg_data = pd.DataFrame({
    'prompt': orig['prompt'],
    'original': orig['text'],
    'located': located['text'],
    'le_llm_edit': llmle['text'],
    'le_llm_edit_masked': llmle_masked['text'],
    'le_mask_infill': mlmle['text'],
    'llm_edit_wo_locate': llmwo['text'],
})

# additionally, read nli class of the original data
orig_nli = read_metric_file('/data/hyeryung/mucoco/new_module/data/logical-consistency/anli-r2-test_prompt_4_below_consistent_threshold_3105.jsonl-results.txt.nli', 'nli')
agg_data['original_nli'] = orig_nli['nli_class']

# random sample 10 rows
sampled_data = agg_data.groupby('prompt').sample(random_state=42, n=1)
sampled_data = sampled_data.sample(random_state=42, n=10)

# check data


In [13]:
agg_data.loc[agg_data['original_nli'] == 'contradiction'].to_excel('/data/hyeryung/mucoco/outputs/20250323_generation_examples/nli_generation_examples_contradiction_all.xlsx', index=False)

In [ ]:
sampled_data.to_excel('/data/hyeryung/mucoco/outputs/20250323_generation_examples/nli_generation_examples.xlsx', index=False)

## Task: Sentiment

In [27]:
# read data
orig = read_outputs('/data/hyeryung/mucoco/new_module/data/sentiment/dev_set_below_positive_threshold_778.jsonl')
llmle = read_outputs('/data/hyeryung/mucoco/outputs/llmedit/results_saehee_2024/qwen/16_pos_edited_38144.jsonl_total_0')
mlmle = read_outputs('/data/hyeryung/mucoco/outputs/sentiment/positive_gpt2/kvi2h6uo/outputs_epsilon0.97.txt.3')
llmwo = read_outputs('/data/hyeryung/mucoco/outputs/llmedit/results_saehee_2024/qwen/18_pos_loc_edit_38398.jsonl')
located = read_outputs('/data/hyeryung/mucoco/outputs/llmedit/dump/iter_loc_edit_qwen/located/16_pos_located_38144.jsonl_filtered_0')
mixmatch = read_outputs('/data/hyeryung/mucoco/outputs/sentiment/mixmatch/clsf_pos_merged/pos.dev_set_below_positive_threshold.jsonl')
bolt = read_outputs('/data/hyeryung/mucoco/outputs/sentiment/bolt/clsf/pos.dev_set_below_positive_threshold.jsonl')
mucola = read_outputs('/data/hyeryung/mucoco/outputs/sentiment/mucola/below_positive_threshold_778/v7ljhn17/outputs_epsilon-5.22.txt')


In [28]:
# additionally, read sequence lengths
orig_all = read_outputs('/data/hyeryung/mucoco/new_module/data/sentiment/dev_set.jsonl')
with open('/data/hyeryung/mucoco/new_module/data/sentiment/dev_set_below_positive_threshold_index.jsonl', 'r') as f:
    index = f.read()
index = [int(i) for i in index.rstrip().split(' ')]
seq_lengths = orig_all.loc[index,'seq_lengths']

In [29]:
# also, read L&E positivity scores ..
mlmle_positivity = read_metric_file('/data/hyeryung/mucoco/outputs/sentiment/positive_gpt2/kvi2h6uo/outputs_epsilon0.97.txt.3-results.txt.sentiment_ext', 
                         'sentiment_ext')

In [35]:

agg_data = pd.DataFrame({
    'prompt': orig['prompt'],
    'original': orig['text'],
    'located': located['text'],
    'le_llm_edit': llmle['text'],
    'le_mask_infill': mlmle['text'],
    'mixmatch': mixmatch['text'],
    'bolt': bolt['text'],
    'mucola': mucola['text'],
    'seq_length': seq_lengths.tolist(),
    'le_mask_infill_positivity': mlmle_positivity
})

# split by length

agg_data_len_12 = agg_data[agg_data['seq_length'] == 12]
agg_data_len_20 = agg_data[agg_data['seq_length'] == 20]
agg_data_len_50 = agg_data[agg_data['seq_length'] == 50]

# random sample 10 rows
sampled_data_len_12 = agg_data_len_12.groupby('prompt').sample(random_state=42, n=1)
sampled_data_len_12 = sampled_data_len_12.sample(random_state=42, n=10)

sampled_data_len_20 = agg_data_len_20.groupby('prompt').sample(random_state=42, n=1)
sampled_data_len_20 = sampled_data_len_20.sample(random_state=42, n=10)

sampled_data_len_50 = agg_data_len_50.groupby('prompt').sample(random_state=42, n=1)
sampled_data_len_50 = sampled_data_len_50.sample(random_state=42, n=10)

# check data


In [36]:

# split by length

agg_data_len_12 = agg_data[agg_data['seq_length'] == 12]
agg_data_len_20 = agg_data[agg_data['seq_length'] == 20]
agg_data_len_50 = agg_data[agg_data['seq_length'] == 50]

# random sample 10 rows
sampled_data_len_12 = agg_data_len_12.groupby('prompt').sample(random_state=42, n=1)
sampled_data_len_12 = sampled_data_len_12.sample(random_state=42, n=10)

sampled_data_len_20 = agg_data_len_20.groupby('prompt').sample(random_state=42, n=1)
sampled_data_len_20 = sampled_data_len_20.sample(random_state=42, n=10)

sampled_data_len_50 = agg_data_len_50.groupby('prompt').sample(random_state=42, n=1)
sampled_data_len_50 = sampled_data_len_50.sample(random_state=42, n=10)

# check data


In [49]:
sampled_data_len_12.to_excel('/data/hyeryung/mucoco/outputs/20250323_generation_examples/positive_generation_examples_len_12.xlsx', index=False)
sampled_data_len_20.to_excel('/data/hyeryung/mucoco/outputs/20250323_generation_examples/positive_generation_examples_len_20.xlsx', index=False)
sampled_data_len_50.to_excel('/data/hyeryung/mucoco/outputs/20250323_generation_examples/positive_generation_examples_len_50.xlsx', index=False)

In [37]:
agg_data_len_12.to_excel('/data/hyeryung/mucoco/outputs/20250323_generation_examples/positive_generation_examples_len_12_all.xlsx', index=False)
agg_data_len_20.to_excel('/data/hyeryung/mucoco/outputs/20250323_generation_examples/positive_generation_examples_len_20_all.xlsx', index=False)
agg_data_len_50.to_excel('/data/hyeryung/mucoco/outputs/20250323_generation_examples/positive_generation_examples_len_50_all.xlsx', index=False)